## Part 1: Preprocessing

In [49]:
# Import our dependencies
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras import layers

#  Import and read the attrition data
attrition_df = pd.read_csv('https://static.bc-edx.com/ai/ail-v-1-0/m19/lms/datasets/attrition.csv')
attrition_df.head()

,Age,Attrition,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,HourlyRate,JobInvolvement,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,Sales,1,2,Life Sciences,2,94,3,...,3,1,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,Research & Development,8,1,Life Sciences,3,61,2,...,4,4,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,Research & Development,2,2,Other,4,92,2,...,3,2,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,Research & Development,3,4,Life Sciences,4,56,3,...,3,3,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,Research & Development,2,1,Medical,1,40,3,...,3,4,1,6,3,3,2,2,2,2


In [50]:
# Determine the number of unique values in each column
attrition_df.nunique()

,0
Age,43
Attrition,2
BusinessTravel,3
Department,3
DistanceFromHome,29
Education,5
EducationField,6
EnvironmentSatisfaction,4
HourlyRate,71
JobInvolvement,4


In [51]:
# Create y_df with the Attrition and Department columns
y_df = attrition_df[['Attrition', 'Department']]




In [52]:
# Create a list of at least 10 column names to use as X data
selected_columns = [
    'Education', 'Age', 'DistanceFromHome', 'JobSatisfaction',
    'OverTime', 'StockOptionLevel', 'WorkLifeBalance',
    'YearsAtCompany', 'YearsSinceLastPromotion', 'NumCompaniesWorked'
]

# Create X_df using your selected columns
X_df = attrition_df[selected_columns]

# Show the data types for X_df
X_df.dtypes

,0
Education,int64
Age,int64
DistanceFromHome,int64
JobSatisfaction,int64
OverTime,object
StockOptionLevel,int64
WorkLifeBalance,int64
YearsAtCompany,int64
YearsSinceLastPromotion,int64
NumCompaniesWorked,int64


In [53]:
# Split the data into training and testing sets
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_df, y_df, test_size=0.2, random_state=42)

In [54]:
# Convert your X data to numeric data types however you see fit
# Add new code cells as necessary

X_df['OverTime'] = X_df['OverTime'].map({'No': 0, 'Yes': 1})
X_train['OverTime'] = X_train['OverTime'].map({'No': 0, 'Yes': 1})
X_test['OverTime'] = X_test['OverTime'].map({'No': 0, 'Yes': 1})



X_df['OverTime'].value_counts()


<ipython-input-54-736e59d68837>:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_df['OverTime'] = X_df['OverTime'].map({'No': 0, 'Yes': 1})


,count
OverTime,
0,1054
1,416


In [55]:
# Create a StandardScaler
scaler = StandardScaler()

# Fit the StandardScaler to the training data
scaler.fit(X_train)


# Scale the training and testing data
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

In [64]:
from sklearn.preprocessing import OneHotEncoder

# Create a OneHotEncoder for the Department column
encoder_dept = OneHotEncoder(sparse_output= False, drop= 'first')

# Fit the encoder to the training data
encoder_dept.fit(y_train[['Department']])

# Create two new variables by applying the encoder
# to the training and testing data
y_train_encoded_dept = encoder.transform(y_train[['Department']])
y_test_encoded_dept = encoder.transform(y_test[['Department']])
y_train_encoded_dept


array([[1., 0.],
       [1., 0.],
       [0., 1.],
       ...,
       [1., 0.],
       [1., 0.],
       [0., 1.]])

In [57]:
# Create a OneHotEncoder for the Attrition column
encoder_attrition = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Fit the encoder to the training data
encoder_attrition.fit(y_train[['Attrition']])

# Create two new variables by applying the encoder
# to the training and testing data
y_train_encoded = encoder_attrition.transform(y_train[['Attrition']])
y_test_encoded = encoder_attrition.transform(y_test[['Attrition']])
y_train_encoded


array([[1., 0.],
       [1., 0.],
       [1., 0.],
       ...,
       [0., 1.],
       [1., 0.],
       [1., 0.]])

## Part 2: Create, Compile, and Train the Model

In [65]:
# Find the number of columns in the X training data.
input_dim = X_train_scaled.shape[1]

# Create the input layer
input_layer = layers.Input(shape=(input_dim,))

# Create at least two shared layers
hidden_layer_1 = layers.Dense(64, activation='relu')(input_layer)
hidden_layer_2 = layers.Dense(32, activation='relu')(hidden_layer_1)


In [66]:
# Create a branch for Department
# with a hidden layer and an output layer

# Create the hidden layer
dept_hidden_layer = layers.Dense(16, activation='relu')(hidden_layer_2)

# Create the output layer
dept_output_layer = layers.Dense(3, activation='softmax', name='department_output')(dept_hidden_layer)


In [67]:
# Create a branch for Attrition
# with a hidden layer and an output layer

# Create the hidden layer
attrition_hidden_layer = layers.Dense(16, activation='relu')(hidden_layer_2)

# Create the output layer
attrition_output_layer = layers.Dense(1, activation='sigmoid', name='attrition_output')(attrition_hidden_layer)


In [68]:
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# Create the model
model = Model(inputs=input_layer, outputs=[attrition_output_layer, dept_output_layer])

# Compile the model
model.compile(optimizer=Adam(learning_rate=0.001),
              loss={'attrition_output': 'binary_crossentropy',
                    'department_output': 'categorical_crossentropy'},
              metrics={'attrition_output': 'accuracy',
                       'department_output': 'accuracy'})

# Summarize the model
model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1             │ (None, 10)             │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_4 (Dense)           │ (None, 64)             │            704 │ input_layer_1[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_5 (Dense)           │ (None, 32)             │          2,080 │ dense_4[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_7 (Dense)           │ (None, 16)             │            528 │ dense_5[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_6 (Dense)           │ (None, 16)             │            528 │ dense_5[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ attrition_output (Dense)  │ (None, 1)              │             17 │ dense_7[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ department_output (Dense) │ (None, 3)              │             51 │ dense_6[0][0]          │
└───────────────────────────┴────────────────────────┴────────────────┴────────────────────────┘

 Total params: 3,908 (15.27 KB)

 Trainable params: 3,908 (15.27 KB)

 Non-trainable params: 0 (0.00 B)

In [71]:
# Train the model
history = model.fit(
    X_train_scaled,
    {'attrition_output': y_train_encoded, 'department_output': y_train_encoded_dept},
    validation_data=(X_test_scaled, {'attrition_output': y_test_encoded, 'department_output': y_test_encoded_dept}),
    epochs=100,
    batch_size=32
)



Epoch 1/100


ValueError: Attr 'Toutput_types' of 'OptionalFromValue' Op passed list of length 0 less than minimum 1.

In [72]:
# Evaluate the model with the testing data
model_loss, model_attrition_accuracy, model_department_accuracy = model.evaluate(
    X_test_scaled,
    {'attrition_output': y_test_encoded, 'department_output': y_test_encoded_dept}
)

ValueError: Arguments `target` and `output` must have the same shape. Received: target.shape=(None, 2), output.shape=(None, 3)

In [73]:
# Evaluate the model on the test data
evaluation_results = model.evaluate(
    X_test_scaled,
    {'attrition_output': y_test_encoded, 'department_output': y_test_encoded_dept},
    verbose=0  # Suppress detailed output
)

# Extract loss and accuracy values
model_loss = evaluation_results[0]  # Total loss
attrition_accuracy = evaluation_results[1]  # Accuracy for Attrition output
department_accuracy = evaluation_results[2]  # Accuracy for Department output

# Print the accuracy results
print(f"Attrition Accuracy: {attrition_accuracy:.4f}")
print(f"Department Accuracy: {department_accuracy:.4f}")


ValueError: Arguments `target` and `output` must have the same shape. Received: target.shape=(None, 2), output.shape=(None, 3)

# Summary

In the provided space below, briefly answer the following questions.

1. Is accuracy the best metric to use on this data? Why or why not?

2. What activation functions did you choose for your output layers, and why?

3. Can you name a few ways that this model might be improved?

YOUR ANSWERS HERE

1.
2.
3.